# Milvus Vector Database Workshop
### Mastering Semantic Search with Milvus Lite

 This notebook is designed to run seamlessly on **Google Colab** using **Milvus Lite**.

**Workshop Agenda:**
1. **Environment Setup**: Installing Milvus Lite and dependencies.
2. **The Fundamentals**: Understanding Chunking, Embeddings, and Indexing.
3. **Data Preparation**: Different approaches to data cleaning and structuring.
4. **Initializing Milvus Collection**: Mastering schema parameters and pros/cons.
5. **Vectorization and Ingestion**: Batching and Streaming for large-scale data.
6. **Mastering Indexing**: Strategies for Milvus Lite and performance tuning.
7. **Semantic & Hybrid Search**: Finding the needle in the haystack.
8. **Data Retrieval**: How Milvus returns your original information.

### Link to Milvus documentation.
https://milvus.io/docs

## 1. Environment Setup
We use `pymilvus[milvus_lite]` which allows us to run Milvus directly inside this notebook without needing a Docker container or a separate server.

In [ ]:
# Install dependencies
!pip install -q pymilvus[milvus_lite] sentence-transformers pandas numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.3/55.3 MB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 301.2/301.2 kB 24.2 MB/s eta 0:00:00


In [ ]:
# Install the Hugging Face Datasets library
!pip install -q datasets

In [ ]:
import os
import itertools
import tensorflow as tf
from datasets import load_dataset
import pandas as pd
import numpy as np
from pymilvus import MilvusClient, DataType
from sentence_transformers import SentenceTransformer


# Cleanup old database if it exists
if os.path.exists("milvus_demo.db"):
    os.remove("milvus_demo.db")


## 2. Deep Dive: The Theory

### A. Understanding Chunking
Before we can turn text into vectors, we must break it into manageable pieces. This is called **Chunking**.

| Chunking Type | Description | Pros | Cons |
| :--- | :--- | :--- | :--- |
| **Fixed-Size** | Splits at exactly X characters/tokens. | Simple, computationally cheap. | Often cuts sentences in half, losing context. |
| **Recursive** | Splits by characters (like paragraphs, then sentences). | Maintains semantic boundaries better. | Requires tuning separators. |
| **Semantic** | Splits where the "meaning" of the text changes. | Highest quality retrieval. | Very slow/expensive (requires LLM/Embeddings). |

### B. Different Embedding Models
Embeddings convert text into a list of numbers (vectors) where similar meanings are close together.

| Model | Origin | Dimension | Pros | Cons |
| :--- | :--- | :--- | :--- | :--- |
| **all-MiniLM-L6-v2** | HuggingFace | 384 | Extremely fast, low memory. | Lower accuracy on complex reasoning. |
| **all-mpnet-base-v2** | HuggingFace | 768 | High accuracy, standard for RAG. | Slower than MiniLM. |
| **text-embedding-3-small** | OpenAI | 1536 | SOTA performance, handles long text. | Paid API, data leaves your server. |

### C. How Hybrid Search Works
Hybrid search combines multiple search techniques to improve accuracy.
1. **Semantic Search (Dense Vectors)**: Finds conceptually similar items (e.g., searching for "happy" finds "joyful").
2. **Keyword Filtering (Scalar Filtering)**: Filters results based on hard attributes (e.g., `year > 2020` or `sentiment == 'positive'`).
3. **Reciprocal Rank Fusion (RRF)**: A method to combine scores from different search methods (like semantic + keyword) into a single ranked list.

## 3. Data Preparation: Different Approaches

Data prep is the foundation of high-quality search. Here are common approaches:

1.  **Cleaning**: Removing HTML tags, emojis, or excessive whitespace to reduce noise in embeddings.
2.  **Structuring**: Extracting metadata (dates, categories, ratings) during ingestion to enable scalar filtering later.
3.  **Augmentation**: Adding context to chunks (e.g., prepending the document title to every chunk) so the embedding model knows the "subject."

### LINK to Epstein Dataset on hugging face
https://huggingface.co/datasets?other=epstein

In [ ]:

print("Fetching Epstein Case Files from Hugging Face...")

try:
    # 1. Load the dataset using streaming=True to handle the large file without high memory usage
    dataset = load_dataset("theelderemo/FULL_EPSTEIN_INDEX", split='train', streaming=True)

    # 2. Slicing: Efficiently pull the first 500 records from the stream
    # We use itertools.islice because you cannot slice a stream like a standard list
    iterator = itertools.islice(dataset, 2000)
    df = pd.DataFrame(list(iterator))

    # 3. Simple Mapping: Ensure 'id' is renamed to 'filename' for the Research Agent
    # This keeps the code simple by assuming the columns 'id' and 'text' always exist
    df.rename(columns={'id': 'filename'}, inplace=True)

    print(f"Success: Loaded {len(df)} records.")
    print(df[['filename', 'text']].head())

except Exception as e:
    print(f"Load Failed: {e}")


Fetching Epstein Case Files from Hugging Face...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

Success: Loaded 2000 records.
           filename                                               text
0  EFTA00005586.pdf  Grand Jury-NY \nEFTA00005586\n\nGrand Jury-NY ...
1  EFTA00004231.pdf  1MS3 \nFD-340 (Rev. 4-11-03) \nFile Number \nF...
2  EFTA00005578.pdf  Grand Jury-NY \nEFTA00005578\n\nUnited States ...
3  EFTA00004070.pdf  By \nToBe Returned K \nYes \nerNo \nReceipt Gi...
4  EFTA00005569.pdf  • \nEFTA00005569\n\nE41TC 3(18,41: BiUD \nEFTA...


In [ ]:
df.head()

,filename,text
0,EFTA00005586.pdf,Grand Jury-NY \nEFTA00005586\n\nGrand Jury-NY ...
1,EFTA00004231.pdf,1MS3 \nFD-340 (Rev. 4-11-03) \nFile Number \nF...
2,EFTA00005578.pdf,Grand Jury-NY \nEFTA00005578\n\nUnited States ...
3,EFTA00004070.pdf,By \nToBe Returned K \nYes \nerNo \nReceipt Gi...
4,EFTA00005569.pdf,"• \nEFTA00005569\n\nE41TC 3(18,41: BiUD \nEFTA..."


### Different Embedding Models
Embeddings convert text into a list of numbers (vectors) where similar meanings are close together.

| Model | Origin | Dimension | Pros | Cons |
| :--- | :--- | :--- | :--- | :--- |
| **all-MiniLM-L6-v2** | HuggingFace | 384 | Extremely fast, low memory. | Lower accuracy on complex reasoning. |
| **all-mpnet-base-v2** | HuggingFace | 768 | High accuracy, standard for RAG. | Slower than MiniLM. |
| **text-embedding-3-small** | OpenAI | 1536 | SOTA performance, handles long text. | Paid API, data leaves your server. |

### Embedding model token limits
The chunk size must be less than or equal to the model's token limit

In [ ]:
# Initialize Embedding Model
model = SentenceTransformer('all-MiniLM-L6-v2') #token limit 512
num_params = sum(p.numel() for p in model.parameters())
print(f"Loaded {len(df)} rows and model with {num_params:,} parameters.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Loaded 2000 rows and model with 22,713,216 parameters.


## 4. Initializing Milvus Collection: Parameters & Pros/Cons

A Collection is a container for your data, similar to a Table in SQL.

### Available Parameters in `create_collection`:
*   `dimension`: Number of elements in the vector (e.g., 384 for MiniLM).
*   `auto_id`: If `True`, Milvus automatically assigns a unique ID to each record.
*   `enable_dynamic_field`: If `True`, you can insert any metadata without pre-defining it in a schema (**Pros**: Extreme flexibility. **Cons**: No strict type checking).
*   `metric_type`: How distance is measured (e.g., `L2` for Euclidean, `COSINE` for angle similarity).
*   `consistency_level`: Controls when data becomes searchable after insertion (e.g., `Strong`, `Bounded`, `Session`, `Eventually`).

### Schema-based vs. Dynamic Schema
*   **Dynamic (Current approach)**: Just provide a dimension and start inserting. Great for prototyping.
*   **Explicit Schema**: Define every field and its type. Recommended for production to ensure data integrity.

In [ ]:
# Updated for Epstein Investigation
COLLECTION_NAME = "+"
client = MilvusClient("milvus_demo.db")

# Clear old data if it exists
if client.has_collection(COLLECTION_NAME):
    client.drop_collection(COLLECTION_NAME)

# Create the new collection
client.create_collection(
    collection_name=COLLECTION_NAME,
    dimension=384,             # Matches all-MiniLM-L6-v2
    auto_id=True,              # Milvus handles IDs for us
    enable_dynamic_field=True, # Allows us to save 'filename' and 'text' easily
    metric_type="COSINE"       # Best for finding "themes" in legal documents
)

print(f"Collection '{COLLECTION_NAME}' created with COSINE metric.")

Collection 'epstein_case_files' created with COSINE metric.


### example of explicit schema

In [ ]:

# COLLECTION_NAME = "fixed_case_files"

# # 2. Define the Blueprint (Schema)
# # We set enable_dynamic_field=False to enforce the fixed schema
# schema = client.create_schema(
#     auto_id=True,
#     enable_dynamic_field=False
# )

# # 3. Add Explicit Fields
# # Primary Key (Unique ID for every row)
# schema.add_field(field_name="pk", datatype=DataType.INT64, is_primary=True)

# # Vector Field (The "Brain" data - matches all-MiniLM-L6-v2)
# schema.add_field(field_name="vector", datatype=DataType.FLOAT_VECTOR, dim=384)

# # Scalar Fields (The Metadata)
# schema.add_field(field_name="filename", datatype=DataType.VARCHAR, max_length=500)
# schema.add_field(field_name="text_content", datatype=DataType.VARCHAR, max_length=5000)
# schema.add_field(field_name="page_count", datatype=DataType.INT64)

# # 4. Create the Collection
# if client.has_collection(COLLECTION_NAME):
#     client.drop_collection(COLLECTION_NAME)

# client.create_collection(
#     collection_name=COLLECTION_NAME,
#     schema=schema
# )

## 5. Vectorization and Ingestion: Streaming vs. Batching

When handling large files, you cannot load everything into memory.

1.  **Batch Ingestion**: Grouping records into batches (e.g., 100-1000) before calling `client.insert`. This reduces the number of network/disk calls.
2.  **Streaming Ingestion**: Using a generator to read a large CSV line by line, vectorizing on the fly, and inserting in small chunks.

Below is a demonstration of **Batch Ingestion**, which is the best balance of speed and memory usage.

### Understanding Chunking
Before we can turn text into vectors, we must break it into manageable pieces. This is called **Chunking**.

| Chunking Type | Description | Pros | Cons |
| :--- | :--- | :--- | :--- |
| **Fixed-Size** | Splits at exactly X characters/tokens. | Simple, computationally cheap. | Often cuts sentences in half, losing context. |
| **Recursive** | Splits by characters (like paragraphs, then sentences). | Maintains semantic boundaries better. | Requires tuning separators. |
| **Semantic** | Splits where the "meaning" of the text changes. | Highest quality retrieval. | Very slow/expensive (requires LLM/Embeddings). |
Below is a demonstration of **Fixed-size**

In [ ]:
CHUNK_SIZE = 1000
OVERLAP     = 200
BATCH_SIZE  = 100

# --- Step 1: Clean Data ---
def clean_investigative_text(input_tensor):
    # Replace newlines, tabs, and pipes with a single space
    clean = tf.strings.regex_replace(input_tensor, r'[\n\r\t|]', ' ')
    # Collapse multiple spaces into one
    clean = tf.strings.regex_replace(clean, r'\s+', ' ')
    return [t.decode('utf-8') for t in clean.numpy()]

# Remove any rows where the 'text' column is empty
df = df.dropna(subset=['text'])

cleaned_texts = clean_investigative_text(tf.constant(df['text'].values))

 # --- Step 2: Chunking Function with Overlap ---
def chunk_text(text, chunk_size=CHUNK_SIZE, overlap=OVERLAP):
      """
      Splits text into overlapping fixed-size chunks.

      Example with chunk_size=10, overlap=3, text="ABCDEFGHIJKLMNOP":
        Chunk 0: ABCDEFGHIJ        (start=0,  end=10)
        Chunk 1: HIJKLMNOPQ        (start=7,  end=17)  <-- 3-char overlap with chunk 0
        Chunk 2: OPQRSTUVWX        (start=14, end=24)  <-- 3-char overlap with chunk 1

      The step between chunks is (chunk_size - overlap).
      """
      chunks = []
      step   = chunk_size - overlap
      start  = 0
      while start < len(text):
          chunks.append(text[start : start + chunk_size])
          start += step
      return chunks

  # --- Step 3: Vectorize + Rolling Batch Insert ---
print("Chunking, vectorizing, and ingesting with overlap...")

data_to_insert = []
total_chunks   = 0

for i, text in enumerate(cleaned_texts):
      filename = df.iloc[i]['filename']
      chunks   = chunk_text(text)

      for chunk_idx, chunk in enumerate(chunks):
          vector = model.encode(chunk).tolist()   # chunk is <= 1000 chars (~256 tokens)

          data_to_insert.append({
              "vector":      vector,
              "filename":    filename,
              "chunk_index": chunk_idx,           # which chunk of this file matched
              "text_excerpt": chunk               # the actual chunk text stored
          })

          # Flush when the batch is full — avoids accumulating everything in memory
          if len(data_to_insert) >= BATCH_SIZE:
              client.insert(collection_name="epstein_case_files", data=data_to_insert)
              total_chunks  += len(data_to_insert)
              print(f"  Inserted batch | running total: {total_chunks} chunks")
              data_to_insert = []

# Flush any remaining records after the loop
if data_to_insert:
      client.insert(collection_name="epstein_case_files", data=data_to_insert)
      total_chunks += len(data_to_insert)

print(f"\nDone. Total chunks ingested: {total_chunks}")
print(f"Original documents: {len(cleaned_texts)} | Avg chunks/doc: {total_chunks /
len(cleaned_texts):.1f}")




Chunking, vectorizing, and ingesting with overlap...
  Inserted batch | running total: 100 chunks
  Inserted batch | running total: 200 chunks
  Inserted batch | running total: 300 chunks
  Inserted batch | running total: 400 chunks
  Inserted batch | running total: 500 chunks
  Inserted batch | running total: 600 chunks
  Inserted batch | running total: 700 chunks
  Inserted batch | running total: 800 chunks
  Inserted batch | running total: 900 chunks
  Inserted batch | running total: 1000 chunks
  Inserted batch | running total: 1100 chunks
  Inserted batch | running total: 1200 chunks
  Inserted batch | running total: 1300 chunks
  Inserted batch | running total: 1400 chunks
  Inserted batch | running total: 1500 chunks
  Inserted batch | running total: 1600 chunks
  Inserted batch | running total: 1700 chunks
  Inserted batch | running total: 1800 chunks
  Inserted batch | running total: 1900 chunks
  Inserted batch | running total: 2000 chunks
  Inserted batch | running total: 21

## 6. Mastering Indexing for Milvus Lite

### What works with Milvus Lite?
Milvus Lite supports major indexing algorithms like `FLAT`, `IVF_FLAT`, and `HNSW`. Since Milvus Lite runs on your local CPU/Memory, **HNSW** is usually the best choice because it provides the best latency-to-recall ratio. **HNSW** does not work in milvus_lite

### Strategy based on Use Case:
| Use Case | Strategy | Index Type |
| :--- | :--- | :--- |
| **Small Dataset (< 5k)** | Brute force is fast enough. | `FLAT` |
| **Low Latency / Web App** | Fast retrieval via graphs. | `HNSW` (M=16, efConstruction=64) |
| **Memory Constrained** | Grouping vectors into buckets. | `IVF_FLAT` (nlist=1024) |
| **Billions of records** | Compression/Quantization. | `IVF_PQ` |

In [ ]:
# Create index for fast searching
index_params = client.prepare_index_params()

index_params.add_index(
    field_name="vector",
    index_type="IVF_FLAT", # Lite version works best with FLAT or IVF_FLAT
    metric_type="COSINE"
)

client.create_index(collection_name="epstein_case_files", index_params=index_params)
print("Index created successfully.")

Index created successfully.


## 7. Search vs. Hybrid Search: The Difference

1.  **Standard Semantic Search**: Finds the top results based ONLY on vector similarity. It might return a negative review if it uses similar descriptive language to your positive query.
2.  **Hybrid Search (Vector + Scalar)**: Finds results that are semantically similar AND match a metadata condition. This is far more powerful for real-world apps where you need to filter by user ID, date, or category.

### How Hybrid Search Works
Hybrid search combines multiple search techniques to improve accuracy.
1. **Semantic Search (Dense Vectors)**: Finds conceptually similar items (e.g., searching for "happy" finds "joyful").
2. **Keyword Filtering (Scalar Filtering)**: Filters results based on hard attributes (e.g., `year > 2020` or `sentiment == 'positive'`).
3. **Reciprocal Rank Fusion (RRF)**: A method to combine scores from different search methods (like semantic + keyword) into a single ranked list.

In [ ]:
# Use an investigative query
query_text = "Royal Family Members"
query_vector = model.encode(query_text).tolist()

results = client.search(
    collection_name="epstein_case_files",
    data=[query_vector],
    limit=3,
    output_fields=["filename", "text_excerpt"] # This is the "Hydration" step
)

print(f"\n--- Top Results for: '{query_text}' ---")

for hit in results[0]:
    print(f"\n📄 FILE: {hit['entity']['filename']}")
    print(f"🔍 MATCH SCORE: {hit['distance']:.4f}") #COSINE similarity, the distance score is a value between 0 and 1, where 1.0 is a perfect match
    print(f"CONTENT: {hit['entity']['text_excerpt'][:400]}...")


--- Top Results for: 'Royal Family Members' ---

📄 FILE: EFTA00002907.pdf
🔍 MATCH SCORE: 0.3626
CONTENT: t)i\orcc k nu bar to throne says Prince Old EFTA00002907 ...

📄 FILE: EFTA00004577.pdf
🔍 MATCH SCORE: 0.3379
CONTENT: nton/Morocco20.JPG Morroco king wedding Clinton/Morocco23.JPG Morroco king wedding EFTA00004618 Clinton/Morocco24.JPG Morroco king wedding • Clinton/Morocco27.JPG Morroco king wedding Clinton/Morocco30.JPG Morroco king wedding 33 St Trop/Clinton Morroco. Nude Clinton/Morocco25.JPG Morroco king wedding Clinton/Morocco28.JPG Morroco king wedding Clinton/Morocco31.J PG Morroco king wedding Clinton/Mo...

📄 FILE: EFTA00004577.pdf
🔍 MATCH SCORE: 0.3037
CONTENT: A00004616 I rr S 3 Clinton/Morocco06.JPG Morroco king wedding Clinton/Morocco09.JPG Morroco king wedding Clinton/Morocco12.JPG Morroco king wedding 31 St Trop/Clinton Morrow. Nude a Clinton/Morocco07.JPG Morroco king wedding Clinton/MoroccolO.JPG Morroco king wedding Clinton/Morocco14.JPG Morroco king wedding Clin

## 8. Data Retrieval: How Original Data is Returned

By default, Milvus only returns the `id` and the `distance` (similarity score) of the match. To get your original text or metadata back, you must use the `output_fields` parameter in the `search` call.

### How it works:
1.  **Storage**: Milvus stores your metadata fields (like `text` or `sentiment`) in its own storage engine.
2.  **Request**: When you search, you specify `output_fields=["text", "rating"]`.
3.  **Hydration**: Milvus finds the matching IDs, fetches the associated metadata for those IDs, and "hydrates" the result object before sending it back to you.

**Pro Tip**: For very large metadata (like full books), it's often better to store only a `document_id` in Milvus and keep the actual text in a fast NoSQL database like MongoDB or Redis.